In [ ]:
import sys
sys.path.append("..")

import pandas as pd
from scrapers import barbora, rimi
from scrapers.common import save_rows

In [ ]:
import sys
sys.path.append("..")

%load_ext autoreload
%autoreload 2

import re
import json
from bs4 import BeautifulSoup
from scrapers.common import get_html

url = "https://www.rimi.lt/e-parduotuve/lt/produktai/mesa-ir-zuvis-/c/SH-9"

params = {
    "currentPage": 1,
    "pageSize": 40,
    "query": ":relevance:allCategories:SH-9:assortmentStatus:inAssortment"
}

html = get_html(url, params=params)
soup = BeautifulSoup(html, "html.parser")

cards = soup.select("li.product-grid__item")

print(f"Found {len(cards)} products")

In [ ]:
DISCOUNT = re.compile(r"-\s*(\d+)\s*%")

promo_cards = []

for card in cards:
    text = card.get_text(" ", strip=True)

    if DISCOUNT.search(text):
        promo_cards.append(card)

print(f"Found {len(promo_cards)} discounted products")

In [ ]:
card = promo_cards[0]

print(card.get_text(" ", strip=True))

In [ ]:
text = card.get_text(" ", strip=True)

match = DISCOUNT.search(text)

discount_pct = int(match.group(1)) if match else None

print("Discount:", discount_pct)

In [ ]:
price_label = card.select_one(".price-label")

print(price_label.prettify()[:3000])

In [ ]:
price_text = price_label.get_text(" ", strip=True)

print(price_text)

In [ ]:
PRICE_PATTERN = re.compile(r"(\d+)\s+(\d{2})\s*€")

price_match = PRICE_PATTERN.search(price_text)

promo_price = (
    float(f"{price_match.group(1)}.{price_match.group(2)}")
    if price_match
    else None
)

print("Promotional price:", promo_price)

In [ ]:
rimi_card_label = card.select_one(
    'img[src*="promotion-labels/rimi-card"]'
)

requires_rimi_card = rimi_card_label is not None

print("Requires Rimi card:", requires_rimi_card)

In [ ]:
for card in promo_cards:
    text = card.get_text(" ", strip=True)

    info_tag = card.select_one("div[data-gtm-eec-product]")
    info = json.loads(info_tag["data-gtm-eec-product"])

    discount_match = DISCOUNT.search(text)
    discount_pct = (
        int(discount_match.group(1))
        if discount_match else None
    )

    price_label = card.select_one(".price-label")
    price_text = price_label.get_text(" ", strip=True) if price_label else ""

    price_match = PRICE_PATTERN.search(price_text)
    promo_price = (
        float(f"{price_match.group(1)}.{price_match.group(2)}")
        if price_match else None
    )

    requires_card = card.select_one(
        'img[src*="promotion-labels/rimi-card"]'
    ) is not None

    print({
        "name": info["name"],
        "gtm_price": info["price"],
        "promo_price": promo_price,
        "discount_pct": discount_pct,
        "requires_rimi_card": requires_card,
    })